# Thelerine 2.0 — warp-based VTON training on Colab

Two stages, run in order:

1. **Warp** — learns where the garment goes (TPS + residual flow). ~10 min on a T4.
2. **Compose** — learns the copy/synthesise blend. ~30 min on a T4.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. Without a GPU
this still runs, but stage 2 takes many hours.

**Run the cells in order the first time.** Later cells depend on variables the
earlier ones define, and will tell you so rather than failing obscurely.

Checkpoints go straight to Drive and both training cells resume automatically,
so if Colab disconnects just re-run from the top — nothing is lost.

## 1. Check the GPU

In [ ]:
import torch

print('torch', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')

## 2. Mount Drive and clone the repo

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/kmafatshe/Thelerine2.0_vton-warp.git'
REPO_DIR = '/content/Thelerine2.0_vton-warp'

import os, subprocess, sys

if os.path.isdir(REPO_DIR):
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                         capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                         capture_output=True, text=True).stderr)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
# Every script below is launched through this helper rather than through `!`.
# IPython's `!cmd {VAR}` substitution fails *silently* when VAR is undefined —
# it passes the literal text `{VAR}` to the program — which turns "you skipped a
# cell" into a confusing argparse error. A plain function raises NameError at
# the point of the mistake instead.
import shlex
import subprocess
import sys


def run(*command):
    """Run a command, streaming its output. Raises if it fails."""
    command = [str(part) for part in command]
    print('$', ' '.join(shlex.quote(part) for part in command), '\n')
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code:
        raise RuntimeError(f'command exited with status {code}')


def show(path):
    from IPython.display import Image, display
    display(Image(str(path)))

## 3. Point at the dataset

Folder names are resolved automatically — `cond/`, `seg/`, `parse/`,
`human_parsing_maps/` and similar all map onto the right role. The cell below
prints what was found, including the **key** each filename reduces to.

Cross-folder matching compares those keys, so if a person file and its parse map
show different keys they will never pair up — that is the first thing to check
if anything comes back unmatched.

In [ ]:
from pathlib import Path

from vtonwarp.data.manifest import describe_layout, resolve_layout

DATA_ROOT = Path('/content/drive/MyDrive/thelerineAI/Itekanye1.1_TripletDataset')
OUTPUT_ROOT = Path('/content/drive/MyDrive/thelerine_ai_outputs/vton_warp')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

assert DATA_ROOT.exists(), f'not found: {DATA_ROOT}'

print(describe_layout(DATA_ROOT))
print('\nresolved roles:')
for role, name in resolve_layout(DATA_ROOT).items():
    print(f'  {role:<18} -> {name or "(NOT FOUND)"}')

In [ ]:
# Fill these in only if the resolved roles above are wrong or missing.
OVERRIDES = {
    'person_dir': None,
    'garment_dir': None,
    'cihp_dir': None,          # e.g. 'cond' or 'human_parsing_maps'
    'segmentation_dir': None,  # e.g. 'seg'
}

# Both set in section 4a, once the diagnostic has answered them.
LABEL_SCHEME = None    # 'cihp' or 'atr'
PARSE_SOURCE = None    # 'cihp' or 'segmentation' — which folder holds the
                       # label maps. A dataset can carry two conditional
                       # folders where only one of them is one.


# Both argument lists are rebuilt from scratch on every call. An earlier
# version appended to a global list, so re-running a cell silently passed the
# same flag twice.
def data_args():
    """Config overrides for train_warp.py / train_tryon.py."""
    args = [f'data.root={DATA_ROOT}']
    args += [f'data.{role}={name}' for role, name in OVERRIDES.items() if name]
    if LABEL_SCHEME:
        args.append(f'data.label_scheme={LABEL_SCHEME}')
    if PARSE_SOURCE:
        args.append(f'data.parse_source={PARSE_SOURCE}')
    return args


def check_args():
    """The same choices, as flags for check_dataset.py."""
    args = []
    for role, name in OVERRIDES.items():
        if name:
            args += [f"--{role.replace('_dir', '-dir')}", name]
    if LABEL_SCHEME:
        args += ['--label-scheme', LABEL_SCHEME]
    if PARSE_SOURCE:
        args += ['--parse-source', PARSE_SOURCE]
    return args


def require_scheme():
    if not (LABEL_SCHEME and PARSE_SOURCE):
        raise RuntimeError(
            'LABEL_SCHEME / PARSE_SOURCE are not set. Run section 4a, then put '
            'both answers into the cell there.'
        )


print('training overrides:', ' '.join(data_args()))
print('check flags       :', ' '.join(check_args()) or '(none)')

## 4a. Which folder holds the parse map, and which convention does it use?

Two silent failures, both answered here.

**Which folder.** A dataset can carry two conditional folders where only one
is a label map — the other being pose heatmaps or one-hot probabilities, which
argmax turns into a perfectly well-formed parse map that describes nothing.

**Which convention.** Under CIHP id 5 is *upper clothes*; under ATR it is
*skirt*, and id 11 is *face* rather than *scarf*. Assume the wrong one and the
model erases the person's face instead of their shirt.

Every combination is scored. Read the `contents:` lines in the layout report
from section 3 alongside this — they say what each folder actually holds.
**Copy both answers into the next cell.**

In [ ]:
run(sys.executable, 'scripts/check_dataset.py',
    '--root', DATA_ROOT, *check_args(), '--diagnose-labels')

In [ ]:
# Both values come from the diagnosis above — copy them exactly.
LABEL_SCHEME = 'cihp'          # 'cihp' or 'atr'
PARSE_SOURCE = 'segmentation'  # 'cihp' or 'segmentation'

print('check flags       :', ' '.join(check_args()))
print('training overrides:', ' '.join(data_args()))

## 4b. Sanity-check the data — do not skip this

One mismatched parse map is a whole percent of a small dataset. Three columns
decide whether training can possibly work:

* **`agnostic`** — the swapped garment must be *completely* gone. On a trouser
  sample the top correctly stays visible; only the trousers should disappear.
* **`garment mask`** — must cover the garment only. If it covers the whole
  frame, background gets warped onto the body.
* **`target garment`** — must show the *same* garment as the `garment` column.
  If it shows different clothing, the label scheme or the per-sample garment
  selection is wrong, and stage 1 will train against the wrong region.

In [ ]:
require_scheme()
run(sys.executable, 'scripts/check_dataset.py',
    '--root', DATA_ROOT, *check_args(),
    '--samples', 6, '--out', '/content/dataset_check.png')

In [ ]:
show('/content/dataset_check.png')

If `segmentation/` turned out to hold a person silhouette rather than a mask of
the flat garment (the check prints its best guess), add
`'segmentation_role': 'ignore'` handling by setting
`OVERRIDES['segmentation_dir'] = None` and passing
`data.segmentation_role=ignore` in the training cells. The garment mask is then
segmented from the product shot's background instead, which works for any
uniform backdrop.

## 5. Stage 1 — train the warper

Watch `warp/shape` fall. It is the silhouette agreement between the warped
garment and the region it has to fill — the number that says whether the
geometry is being learned.

Set `FRESH_START = True` if you are retraining after fixing a data problem;
otherwise the run resumes from the old, wrongly-trained checkpoint.

In [ ]:
WARP_OUT = OUTPUT_ROOT / 'warp'
WARP_STEPS = 12000
BATCH_SIZE = 8
FRESH_START = False

require_scheme()
run(sys.executable, 'train_warp.py', '--config', 'configs/warp.yaml',
    *data_args(),
    f'output_dir={WARP_OUT}',
    f'train.steps={WARP_STEPS}',
    f'train.batch_size={BATCH_SIZE}',
    f'train.resume={str(not FRESH_START).lower()}',
    'train.num_workers=2',
    'train.sample_every=1000',
    'train.save_every=1000')

In [ ]:
# Columns: garment | agnostic | coarse | warped | overlay | target | gt | flow.
# `overlay` is the one that matters — the garment should sit on the body in the
# right place and shape before you start stage 2.
show(sorted((WARP_OUT / 'samples').glob('*.png'))[-1])

## 6. Stage 2 — train the composer

The warper is frozen here. Watch the `alpha` column in the samples: it should be
bright and crisp over the garment, meaning the model is *copying* real garment
pixels rather than hallucinating them.

In [ ]:
TRYON_OUT = OUTPUT_ROOT / 'tryon'
TRYON_STEPS = 15000

require_scheme()
run(sys.executable, 'train_tryon.py', '--config', 'configs/tryon.yaml',
    *data_args(),
    f'output_dir={TRYON_OUT}',
    f'train.warp_checkpoint={WARP_OUT}/warp.pt',
    f'train.steps={TRYON_STEPS}',
    f'train.batch_size={BATCH_SIZE}',
    f'train.resume={str(not FRESH_START).lower()}',
    'train.num_workers=2',
    'train.sample_every=1000',
    'train.save_every=1000')

In [ ]:
show(sorted((TRYON_OUT / 'samples').glob('*.png'))[-1])

## 7. Optional — adversarial sharpening pass

Only once stage 2 has converged and the output is structurally correct but soft.
It writes to a **separate** folder so a GAN collapse cannot destroy the
checkpoint you already have.

In [ ]:
import shutil

GAN_OUT = OUTPUT_ROOT / 'tryon_gan'
GAN_OUT.mkdir(parents=True, exist_ok=True)
if not (GAN_OUT / 'tryon.pt').exists():
    shutil.copy(TRYON_OUT / 'tryon.pt', GAN_OUT / 'tryon.pt')

run(sys.executable, 'train_tryon.py', '--config', 'configs/tryon.yaml',
    *data_args(),
    f'output_dir={GAN_OUT}',
    f'train.warp_checkpoint={WARP_OUT}/warp.pt',
    'loss.gan=0.5', 'train.gan_start_step=0', 'train.lr=0.00005',
    f'train.steps={TRYON_STEPS + 4000}',
    f'train.batch_size={BATCH_SIZE}',
    'train.num_workers=2',
    'train.sample_every=500',
    'train.save_every=500')

## 8. Inference — every garment on every person

The honest evaluation. Training is self-paired, so reconstructing someone in
their own clothes proves nothing. **Rows are people, columns are garments —
judge the off-diagonal cells.**

In [ ]:
GRID = OUTPUT_ROOT / 'grid.png'

run(sys.executable, 'infer.py',
    '--checkpoint', TRYON_OUT / 'tryon.pt', '--grid',
    '--root', DATA_ROOT, '--limit', 5, '--out', GRID)
show(GRID)

## Notes

* **Disconnected?** Re-run from the top. Both training cells resume from the
  last checkpoint in Drive. To start clean, set `FRESH_START = True`.
* **Retraining after a data fix?** Set `FRESH_START = True`, or the run will
  resume the checkpoint that was trained against the wrong targets.
* **Want to train longer?** Raise `WARP_STEPS` / `TRYON_STEPS` and re-run — the
  schedule is replayed against the new total, so extending a finished run works.
* **Out of memory?** Lower `BATCH_SIZE` to 4, or add `train.accumulate=2` to
  keep the effective batch while halving memory.
* **Both stages must use the same resolution and label scheme**, or they
  disagree about what the agnostic input means. Stage 2 checks the resolution
  and refuses to start on a mismatch.
* Full tuning and diagnostics tables are in the repo `README.md`.